# SmartScholar AI - Demo Notebook
Runs the complete **fan-out / fan-in** workflow for one sample application.
Run the cells from top to bottom (**Run All**). Open the notebook from the project folder, and select the Python
kernel of the virtual environment where you ran `pip install -r requirements.txt`.

## 1. Load the project configuration

In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())          # the notebook lives in the project root

import config
print("Eligibility rules :", config.ELIGIBILITY_RULES)
print("Score weights     :", config.WEIGHTS)
print("Decision thresholds:", config.THRESHOLDS)
print("GROQ_API_KEY set  :", bool(config.GROQ_API_KEY))

## 2. Initialise the agents
The `Coordinator` loads the rubric into the RAG engine, opens shared memory and the log, and checks whether Groq is available.
If it is not, the system runs in clearly labelled **demo mode** (all scores and decisions are still computed by Python tools).

In [ ]:
from coordinator import Coordinator

coordinator = Coordinator()
print("Mode:", coordinator.mode)
if not coordinator.llm_on:
    print("Reason:", coordinator.llm_message)
print("RAG :", coordinator.rag.mode, "-", len(coordinator.rag.chunks), "rubric clauses indexed")

kernel = coordinator.create_kernel()
agents = coordinator.create_agents(kernel)
for key, agent in agents.items():
    print(f"  {type(agent).__name__:<17} ready")
print("Semantic Kernel plugins registered:", sorted(kernel.plugins))

## 3. Submit a sample application

In [ ]:
sample = {
    "application_id": "DEMO001",
    "student_name": "Rahul Sharma",
    "cgpa": 8.5,
    "family_income": 250000,
    "course": "B.Tech CSE",
    "achievements": ["Hackathon Winner", "Academic Excellence"],
    "statement": "I need financial assistance to continue my education.",
}
sample

## 4. Run the fan-out / fan-in workflow

In [ ]:
record = await coordinator.evaluate_async(sample)     # Jupyter allows top-level await
print("Workflow finished for", record["application_id"])

## 5. Agent execution trace
The Merit and Need agents start together (fan-out); the Panel Manager starts after both finish (fan-in).

In [ ]:
for number, step in enumerate(record["execution_trace"], start=1):
    print(f"{number}. {step['agent']}  [{step['status']}]")
    print("   action :", step["action"])
    print("   tools  :", ", ".join(step["tools_used"]) or "none")
    print("   input  :", step["input_summary"])
    print("   output :", step["output_summary"])
    print("   time   :", step["started"][11:], "->", step["finished"][11:])

## 6. Final weighted score and decision

In [ ]:
final = record["final_decision"]
print("Eligibility :", final["eligibility"])
print("Merit score :", final["merit_score"], "/ 100")
print("Need score  :", final["need_score"], "/ 100")
print("Formula     :", final["formula"])
print("Final score :", final["final_score"])
print("DECISION    :", final["decision"])
print()
print(final["justification"])

## 7. Rubric references
These clauses were retrieved from the rubric (ChromaDB similarity search) and applied by the agents.

In [ ]:
for ref in final["rubric_references"]:
    print(f"{ref['clause_id']:<7} [{ref['section']}]  {ref['title']}")

## 8. Execution log (append-only) for this application

In [ ]:
for entry in coordinator.log.read_all("DEMO001")[-8:]:
    print(entry["timestamp"][11:], "|", entry["agent"], "|", entry["action"], "|", entry["output_summary"])